In [ ]:
import torch
import torch.nn as nn
import onnx

import huggingface_hub
import open_clip
from timm.utils import reparameterize_model

In [ ]:
modelDir = 'models/mobileClip2-s0'

# LOADS

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms('MobileCLIP2-S0', pretrained='dfndr2b')
model.eval()
model = reparameterize_model(model)

In [ ]:
# mobileClip2 uses functionally identical tokenizer to openai
tokenizerPath = huggingface_hub.hf_hub_download(
    repo_id='openai/clip-vit-base-patch32',
    filename='tokenizer.json',
    local_dir=modelDir
)

In [ ]:
imgSz = model.visual.image_size[0]
txtSz = model.text.context_length
embDim = model.text.output_dim

print('image_size:', imgSz)
print('max_position_embeddings:', txtSz)
print('projection_dim:', embDim)

# DEFINE PREPROCESS MODEL

In [ ]:
class Preprocess(nn.Module):
    def __init__(self, size, normalize):
        super().__init__()
        self.norm = normalize
        self.sz = size
        
    def forward(self, x):
        x = torch.reshape(x, (self.sz*self.sz, 3)).T
        x = torch.reshape(x, (3, self.sz, self.sz))
        x = self.norm(x)
        x = x.unsqueeze(0)
        
        return x

In [ ]:
preprocess = Preprocess(imgSz, lambda x: x) # mobileClip doesnt require normalizing the input
preprocess.eval()

# CONVERT MODELS

In [ ]:
txtOnnx = torch.onnx.export(model.text, torch.ones(1, txtSz, dtype=torch.int32))

In [ ]:
preOnnx = torch.onnx.export(preprocess, torch.ones(1, 3*imgSz*imgSz, dtype=torch.float32))

In [ ]:
intOnnx = torch.onnx.export(model.visual, torch.ones(1, 3, imgSz, imgSz, dtype=torch.float32))

In [ ]:
try:
    os.remove(f'{modelDir}/preprocess.onnx')
except:
    pass
preOnnx.save(f'{modelDir}/preprocess.onnx')

In [ ]:
try:
    os.remove(f'{modelDir}/intermediate.onnx')
except:
    pass
intOnnx.save(f'{modelDir}/intermediate.onnx')

In [ ]:
preprocessOnnx = onnx.load(f'{modelDir}/preprocess.onnx')
intermediate = onnx.load(f'{modelDir}/intermediate.onnx')

# COMBINE MODELS

In [ ]:
combinedOnnx = onnx.compose.merge_models(preprocessOnnx, intermediate,
    prefix1='p',
    prefix2='v',
    io_map=[(
    preprocessOnnx.graph.output[0].name,
    intermediate.graph.input[0].name)])

# SAVE MODELS

In [ ]:
try:
    os.remove(f'{modelDir}/textual.onnx')
except:
    pass
txtOnnx.save(f'{modelDir}/textual.onnx')

In [ ]:
try:
    os.remove(f'{modelDir}/visual.onnx')
except:
    pass
onnx.save(combinedOnnx, f'{modelDir}/visual.onnx')

# SETUP JSON CONFIG

In [ ]:
json = f"""
{{
  "mlPath": "",
  "txtInputDType": "int32",
  "visInputDType": "float32",
  "imgSize": {imgSz},
  "contextLen": {txtSz},
  "embeddingLen": {embDim},
  "similarity": "cosineSimilarity"
}}
"""

In [ ]:
with open(f'{modelDir}/mlconfig.json', 'w') as f:
    f.write(json)